# Simple Redis-Py Example
This notebook contains simple redis python commands.  

In [32]:
import redis

Connect to local server -- no hostname or ip is needed

In [33]:
rd = redis.Redis(host='localhost', decode_responses=True)

In [3]:
rd.set('user:101:name', 'pinot')

True

In [4]:
rd.set('user:102:name', 'pinot')

True

In [5]:
rd.set("user:101", "kuy")

True

In [6]:
rd.sadd("test", 'd', 'ss')

2

In [7]:
rd.set('name', 'Alice')

True

In [8]:
rd.scan()

(0, ['test', 'user:102:name', 'name', 'user:101:name', 'user:101'])

In [9]:
rd.get('name')

'Alice'

In [10]:
rd.hset('post:1', 'user', 101)
rd.hset('post:1', 'content', 'this is the first post')

1

In [11]:
rd.hgetall('post:1')

{'user': '101', 'content': 'this is the first post'}

In [12]:
rd.rpush('user:101:post', 1)
rd.rpush('user:101:post', 8)

2

In [13]:
rd.llen('user:101:post')

2

In [14]:
rd.lrange('user:101:post', 0, -1)

['1', '8']

In [15]:
rd.sadd('user:101:follows', 104)
rd.sadd('user:101:follows', 105)

1

In [16]:
rd.scard('user:101:follows')

2

In [17]:
rd.smembers('user:101:follows')

{'104', '105'}

In [29]:
cursor = 0
cursor, keys = rd.scan(cursor=cursor, match='user:*')
while cursor > 0:
    for key in keys:
        print('found: ', key)
    cursor, keys = rd.scan(cursor=cursor, match='username:*')  

for key in keys:
    print('found: ', key)

found:  user:101:follows
found:  user:102:name
found:  user:101
found:  user:101:name
found:  user:101:post


## Summary of commands used:

In [ ]:
# SET / GET
rd.set("user:1:name", "Peerawit")
print(rd.get("user:1:name"))   # 'Peerawit'

# EXISTS
print(rd.exists("user:1:name"))  # 1 = มี, 0 = ไม่มี

# DEL
rd.delete("user:1:name")
print(rd.get("user:1:name"))     # None

rd.set("otp:1234", "987654")
rd.expire("otp:1234", 60)   # หมดอายุใน 60 วินาที

print(rd.ttl("otp:1234"))   # เหลืออีกกี่วินาที


In [ ]:
import json

# เก็บผล query จาก MongoDB หรือ DB อื่นเป็น cache
key = "cache:recipe:cuisine:thai"

result = [
    {"id": 101, "name": "Pad Thai"},
    {"id": 102, "name": "Tom Yum Kung"},
]
rd.set(key, json.dumps(result))

cached = rd.get(key)
recipes = json.loads(cached)
print(recipes[0]["name"])  # 'Pad Thai'

# นับจำนวน request (counter)
rd.incr("stats:page_view:/home") # ไม่มีแต่แรกจะสร้างใหม่ ละเพิ่มทีละ 1
print(rd.get("stats:page_view:/home"))

Pad Thai
1


In [ ]:
# ทำ queue สำหรับ background task
queue_key = "task:queue"

# producer
rd.lpush(queue_key, "send_email:order_123")
rd.lpush(queue_key, "send_email:order_124")
# lpush(key, value)

# consumer (worker)
while True:
    task = rd.rpop(queue_key) 
    if not task:
        break
    print("Processing:", task)

# timeline ของ user
for i in range(10, 100):
    rd.lpush("timeline:user:1", f"post:{i}")
posts = rd.lrange("timeline:user:1", 0, 19)  # 20 โพสต์ล่าสุด
print(posts)


['post:99', 'post:98', 'post:97', 'post:96', 'post:95', 'post:94', 'post:93', 'post:92', 'post:91', 'post:90', 'post:89', 'post:88', 'post:87', 'post:86', 'post:85', 'post:84', 'post:83', 'post:82', 'post:81', 'post:80']


In [ ]:
# เก็บค่าไม่ซ้ำ
bad_words_key = "words:denied"

rd.sadd(bad_words_key, "XXX", "YYY", "ZZZ")

print(rd.sismember(bad_words_key, "XXX"))  # True
print(rd.sismember(bad_words_key, "ABC"))  # False

print(rd.smembers(bad_words_key))  # {'XXX', 'YYY', 'ZZZ'}
print(rd.scard(bad_words_key))     # 3 cardinality


1
0
{'YYY', 'ZZZ', 'XXX'}
3


In [74]:
# ไว้ทำleaderboard, pq
leaderboard_key = "game:leaderboard"

# เพิ่มคะแนน
rd.zadd(leaderboard_key, {"alice": 1500, "bob": 2000})
rd.zadd(leaderboard_key, {"charlie": 1800, "dog":2500})

# Top 3 (คะแนนสูงสุดก่อน)
top3 = rd.zrevrange(leaderboard_key, 0, 2, withscores=True)
print(top3)  # [('dog', 2500.0), ('bob', 2000.0), ('charlie', 1800.0)]

[('dog', 2500.0), ('bob', 2000.0), ('charlie', 1800.0)]


In [75]:
rd.zpopmax(leaderboard_key)  # ('dog', 2500.0)
rd.zpopmin(leaderboard_key)  # ('alice', 1500.0)

[('alice', 1500.0)]